
# **Предобработка: Картинки и Данные (OpenCV, PIL, torchvision, pandas)Быстрая загрузка и конвертация (Без ошибок с цветами):**


In [ ]:
import cv2
from PIL import Image
import numpy as np

# OpenCV: Читаем и сразу чиним цвета (BGR -> RGB)
img_bgr = cv2.imread("data/image.jpg")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# PIL: Загрузка с гарантированным RGB (спасает от ч/б и PNG с альфа-каналом)
img_pil = Image.open("data/image.jpg").convert("RGB")

# **Синхронные аугментации (Картинка + Маска):**

In [ ]:
import torch
from torchvision.transforms import v2

transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomRotation(degrees=15),
    v2.ColorJitter(brightness=0.2), # Применится только к картинке
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# img_tensor, mask_tensor = transform(img_pil, mask_pil)

# **Работа с таблицами (Pandas):**


In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
# Создаем словарь: {путь_к_файлу: метка} для быстрого доступа в Dataset
img_to_label = dict(zip(df['image_path'], df['label']))


# **2. 🧠 Извлечение фичей и Zero-Shot (timm, open_clip, torch)Timm: Достаем эмбеддинги (без классификации):**#


In [ ]:
import timm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# num_classes=0 сносит "голову", модель возвращает векторы
model_cv = timm.create_model('resnet34', pretrained=True, num_classes=0).to(device)
model_cv.eval()

# На входе тензор [Batch, 3, 224, 224], на выходе [Batch, 512]
with torch.no_grad():
    features = model_cv(image_tensor.to(device))

# **OpenCLIP: Классификация текстом (Zero-Shot):**# 

In [ ]:
import open_clip
import torch.nn.functional as F

model_clip, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model_clip.eval().to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Картинка и кандидаты
image_input = preprocess(Image.open("img.jpg")).unsqueeze(0).to(device)
text_inputs = tokenizer(["a healthy brain", "a brain with a tumor"]).to(device)

with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model_clip.encode_image(image_input)
    text_features = model_clip.encode_text(text_inputs)
    
    # Нормализация (защита от NaN)
    image_features = F.normalize(image_features, p=2, dim=-1)
    text_features = F.normalize(text_features, p=2, dim=-1)
    
    # Считаем вероятности
    probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)


# **3. 📝 NLP и Генерация (transformers)Правильный промпт через Chat Template (Для LLM):**


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen1.5-1.8B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

messages = [
    {"role": "system", "content": "You are a helpful AI. Output only JSON."},
    {"role": "user", "content": "Analyze this data..."}
]

# Токенизируем с шаблоном
text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text_prompt, return_tensors="pt").to(device)

# Генерация
with torch.no_grad():
    output_ids = model_llm.generate(
        **inputs, 
        max_new_tokens=100, 
        temperature=0.1, # 0.1 для точных задач
        do_sample=True
    )

# Обрезаем промпт из ответа
response = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# **4. 🚀 Классический ML (План "Б"): xgboost, scikit-learnИспользуй это, когда достал векторы (эмбеддинги) из картинок или текста, и нужно быстро обучить классификатор на нестандартных данных без переобучения тяжелой нейронки.**


In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression

# X - матрица фичей numpy [N_samples, 512], y - метки [N_samples]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost (Мощно, если данных хотя бы пару тысяч)
xgb_model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_val)

# Логистическая регрессия (Идеально для эмбеддингов, если мало данных!)
log_reg = LogisticRegression(max_iter=1000, C=1.0)
log_reg.fit(X_train, y_train)
lr_preds = log_reg.predict(X_val)

print(f"XGB F1: {f1_score(y_val, xgb_preds, average='macro')}")


# **5. ⚙️ База PyTorch: Custom Dataset и Циклы (torch, tqdm)Скелет кастомного датасета (Основа олимпиад):**


In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

class OlympiadDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image) # Для v2 можно передать и маску
            
        return image, torch.tensor(label, dtype=torch.long)

# Использование
dataset = OlympiadDataset(paths_list, labels_list, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)

# Красивый цикл инференса/извлечения с прогресс-баром
all_features = []
for images, _ in tqdm(dataloader, desc="Processing..."):
    images = images.to(device)
    with torch.no_grad():
        features = model_cv(images)
        all_features.append(features.cpu().numpy())

# Собираем все батчи в одну огромную матрицу для XGBoost
X_final = np.vstack(all_features)